[Reference](https://pub.aimind.so/transformers-explained-building-blocks-of-modern-nlp-5a1dcc19c24d)

In [2]:
!pip install pymilvus

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.0/278.0 kB 5.7 MB/s eta 0:00:00


In [4]:
# Minimal RAG baseline with Milvus + sentence-transformers
# Priya's "get it working, then make it fast" version.

import os
import uuid
from typing import List, Tuple

# 1) Embeddings
from sentence_transformers import SentenceTransformer
import numpy as np

# 2) Milvus client
from pymilvus import connections, FieldSchema, CollectionSchema, DataType, Collection, utility

# ---- Config ----
COLLECTION_NAME = "docs_v1"
DIM = 768
INDEX_TYPE = "HNSW"           # try IVF_FLAT for simple baselines, then IVF_PQ or DiskANN
METRIC_TYPE = "IP"            # Inner product (cosine w/ normalization)
EF_CONSTRUCTION = 200
M = 16
EF_SEARCH = 64

# ---- 1) Model ----
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def embed_texts(texts: List[str]) -> np.ndarray:
    vecs = model.encode(texts, normalize_embeddings=True, batch_size=64, convert_to_numpy=True)
    return vecs.astype("float32")

# ---- 2) Connect to Milvus ----
# Assumes local Milvus standalone or a remote instance.
# For managed, use the Zilliz Cloud connection string and token.
connections.connect(alias="default", host="localhost", port="19530")

# ---- 3) Define / create collection ----
fields = [
    FieldSchema(name="id", dtype=DataType.VARCHAR, is_primary=True, auto_id=False, max_length=64),
    FieldSchema(name="doc_id", dtype=DataType.VARCHAR, max_length=64),
    FieldSchema(name="chunk", dtype=DataType.VARCHAR, max_length=2048),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=DIM),
]
schema = CollectionSchema(fields, description="RAG chunks")

if utility.has_collection(COLLECTION_NAME):
    utility.drop_collection(COLLECTION_NAME)
col = Collection(name=COLLECTION_NAME, schema=schema, consistency_level="Bounded")

# Create index before insert for better memory layout
col.create_index(
    field_name="embedding",
    index_params={
        "index_type": INDEX_TYPE,
        "metric_type": METRIC_TYPE,
        "params": {"M": M, "efConstruction": EF_CONSTRUCTION},
    },
)

# ---- 4) Ingest function ----
def ingest(chunks: List[Tuple[str, str]]):
    # chunks: list of (doc_id, text)
    ids = [str(uuid.uuid4()) for _ in chunks]
    texts = [c[1] for c in chunks]
    vecs = embed_texts(texts)
    col.insert([ids, [c[0] for c in chunks], texts, vecs])
    col.flush()

# ---- 5) Search ----
def search(query: str, top_k: int = 5):
    qvec = embed_texts([query])[0].tolist()
    col.load()
    results = col.search(
        data=[qvec],
        anns_field="embedding",
        param={"metric_type": METRIC_TYPE, "params": {"ef": EF_SEARCH}},
        limit=top_k,
        output_fields=["doc_id", "chunk"],
    )
    hits = []
    for hit in results[0]:
        hits.append({
            "score": float(hit.distance),
            "doc_id": hit.entity.get("doc_id"),
            "chunk": hit.entity.get("chunk")
        })
    return hits

# ---- 6) Demo corpus ----
docs = [
    ("d1", "Transformers rely on attention mechanisms to compute contextual token representations."),
    ("d1", "Encoder-only architectures are ideal for semantic search and classification."),
    ("d2", "Decoder-only models excel at generative tasks such as summarization and code synthesis."),
    ("d3", "Hybrid retrieval combines dense and sparse signals to balance semantic match and exact recall."),
    ("d4", "Index parameters like efSearch and M in HNSW directly trade latency for recall."),
]
ingest(docs)

# ---- 7) Try a query ----
for q in ["semantic search architectures", "how attention helps embeddings"]:
    print("---", q)
    for h in search(q, top_k=3):
        print(h["score"], h["doc_id"], h["chunk"])